# Bài tập lớn — Phân tích hồi quy

Đề tài: Phân tích các yếu tố ảnh hưởng đến nồng độ PM2.5 tại TP.HCM.

Câu hỏi nghiên cứu: trong số các biến khí tượng (nhiệt độ, độ ẩm), nồng
độ các chất ô nhiễm khác (TSP, CO, NO2, SO2, O3), vị trí địa lý và thời
gian, yếu tố nào ảnh hưởng đáng kể đến PM2.5? Phương pháp chính là hồi
quy tuyến tính bội. Phần mở rộng phân tích thêm mô hình Y=TSP để hiểu
quan hệ giữa hai loại bụi và đánh giá pattern nguồn phát thải.

Dataset: HealthyAir Project (Data in Brief, 2022).


# Chương 1. Xác định bài toán

## 1.1 Bối cảnh

Bụi mịn PM2.5 là tác nhân ô nhiễm không khí nguy hiểm hàng đầu theo WHO.
TP.HCM với mật độ giao thông cao và nhiều khu công nghiệp đang đối mặt
với vấn đề PM2.5 nghiêm trọng. Hiểu rõ yếu tố nào ảnh hưởng đến PM2.5
là cơ sở để đưa ra chính sách kiểm soát ô nhiễm hiệu quả.

## 1.2 Câu hỏi nghiên cứu

Câu hỏi chính: yếu tố nào ảnh hưởng đáng kể đến PM2.5 và mức độ ảnh
hưởng của từng yếu tố ra sao?

Câu hỏi mở rộng: pattern nguồn phát thải PM2.5 và TSP tại các vị trí
khác nhau có giống nhau không? TSP có thể là proxy đo nhanh thay cho
PM2.5 không?

## 1.3 Đối tượng và phạm vi

Đối tượng: nồng độ PM2.5 theo giờ tại 6 trạm quan trắc HealthyAir trên
địa bàn TP.HCM. Trạm 1 (ĐHQG, Thủ Đức), trạm 2/5/6 (giao thông), trạm 3
(KCN Tân Bình), trạm 4 (dân cư Bình Thạnh).

Phạm vi thời gian: từ 02/2021 đến 06/2022 với tổng 52.548 quan sát giờ.

## 1.4 Tiêu chí thành công

- R²_Adj của OLS đạt ≥ 0.5
- RMSE trên test set ≤ 10 µg/m³
- Chênh lệch R² train vs test ≤ 0.1


## Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import jarque_bera

from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report, accuracy_score
from scipy import stats
from scipy.stats import boxcox, boxcox_normmax, chi2

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.options.display.float_format = '{:.4f}'.format

# Chương 2. Mô tả dữ liệu và phân tích khám phá

## 2.1 Tải dữ liệu

In [ ]:
df = pd.read_csv('Air Quality Ho Chi Minh City.csv')
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
print(f'Số quan sát: {len(df):,}')
print(f"Khoảng thời gian: {df['date'].min()} → {df['date'].max()}")
print(f"Số trạm: {df['Station_No'].nunique()}")
df.head()

## 2.2 Tiền xử lý

### 2.2.1 Xử lý giá trị thiếu

Mỗi trạm có đặc trưng ô nhiễm khác nhau (KCN khác dân cư khác giao thông),
nên ta điền giá trị thiếu bằng median của từng trạm thay vì median toàn cục.

In [ ]:
numeric_cols = ['TSP','PM2.5','O3','CO','NO2','SO2','Temperature','Humidity']

print('Tỷ lệ missing trước xử lý (%):')
print((df[numeric_cols].isnull().sum() / len(df) * 100).round(3))

df[numeric_cols] = df.groupby('Station_No')[numeric_cols].transform(
    lambda x: x.fillna(x.median()))

print(f'\nSau xử lý: {df[numeric_cols].isnull().sum().sum()} missing values')

### 2.2.2 Feature engineering thời gian

Dữ liệu là chuỗi thời gian theo giờ. Ta tạo các biến phái sinh từ cột
date để khai thác các pattern thời gian:

- hour_group: 5 nhóm giờ (đêm, sáng cao điểm, ban ngày, chiều cao điểm, tối)
- is_weekend: 1 nếu thứ Bảy hoặc Chủ nhật
- is_dry_season: 1 nếu tháng 11-4 (mùa khô ở TP.HCM)

In [ ]:
df['hour'] = df['date'].dt.hour
df['dow'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['is_weekend'] = (df['dow'] >= 5).astype(int)
df['is_dry_season'] = df['month'].isin([11,12,1,2,3,4]).astype(int)

def hour_group(h):
    if h < 6: return 'night'
    if h < 10: return 'morning_rush'
    if h < 16: return 'daytime'
    if h < 20: return 'evening_rush'
    return 'evening'
df['hour_group'] = df['hour'].apply(hour_group)

print(df['hour_group'].value_counts())
print(f"\nTỷ lệ cuối tuần: {df['is_weekend'].mean()*100:.1f}%")
print(f"Tỷ lệ mùa khô: {df['is_dry_season'].mean()*100:.1f}%")

### 2.2.3 Phân tích outlier

PM2.5 có giá trị max = 403.69 µg/m³, cao bất thường so với trung vị 17.5.
Ta cần đánh giá liệu đây là lỗi đo hay hiện tượng thực.

In [ ]:
print('Percentile PM2.5:')
for p in [50, 90, 95, 99, 99.5, 99.9]:
    print(f'  {p}%: {df["PM2.5"].quantile(p/100):.2f}')
print(f'  Max: {df["PM2.5"].max():.2f}')

threshold = df['PM2.5'].quantile(0.99)
n_high = (df['PM2.5'] > threshold).sum()
print(f'\nQuan sát > P99 ({threshold:.1f}): {n_high} ({n_high/len(df)*100:.2f}%)')

Ta giữ các quan sát PM2.5 cao này vì đây có thể là các sự kiện ô nhiễm
thực tế (cháy rừng, kẹt xe, thời tiết bất lợi). Loại bỏ sẽ làm mất
thông tin về các tình huống cực đoan vốn là tình huống quan trọng nhất
cần dự báo.

## 2.3 Thống kê mô tả

In [ ]:
desc = df[numeric_cols].describe().T
desc['skewness'] = df[numeric_cols].skew()
desc['kurtosis'] = df[numeric_cols].kurt()
print(desc[['mean','std','min','50%','max','skewness','kurtosis']].round(2))

PM2.5 có skewness = 4.31, kurtosis = 28.6 — phân phối lệch phải mạnh,
đuôi rất dày. TSP cũng lệch phải nhưng nhẹ hơn. Nhiệt độ và độ ẩm có
phân phối gần đối xứng. Phân phối lệch của PM2.5 gợi ý có thể cần biến
đổi log; ta sẽ thử ở mục 3.9.

## 2.4 Phân phối các biến

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flatten(), numeric_cols):
    ax.hist(df[col], bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'{col}  (skew={df[col].skew():.2f})')
plt.tight_layout(); plt.show()

## 2.5 Quan hệ giữa các biến với PM2.5

In [ ]:
features = ['TSP','Temperature','Humidity','CO','NO2','SO2','O3']
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, f in zip(axes.flatten(), features):
    ax.scatter(df[f], df['PM2.5'], s=2, alpha=0.3)
    ax.set_xlabel(f); ax.set_ylabel('PM2.5')
    r = df[[f,'PM2.5']].corr().iloc[0,1]
    ax.set_title(f'r = {r:.3f}')
axes.flatten()[-1].axis('off')
plt.tight_layout(); plt.show()

## 2.6 Ma trận tương quan

In [ ]:
corr = df[numeric_cols].corr()
plt.figure(figsize=(10,7))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', center=0)
plt.title('Ma trận tương quan Pearson')
plt.show()

print('\nTương quan với PM2.5 (xếp theo |r|):')
print(corr['PM2.5'].drop('PM2.5').reindex(
    corr['PM2.5'].drop('PM2.5').abs().sort_values(ascending=False).index
).round(4))

TSP có tương quan dương rất mạnh với PM2.5 (r = 0.66). CO có tương
quan dương trung bình (r = 0.28). Nhiệt độ và độ ẩm có tương quan âm
nhẹ với PM2.5.

## 2.7 Biến động theo thời gian

### 2.7.1 PM2.5 theo ngày tại các trạm

In [ ]:
df['date_only'] = df['date'].dt.date
daily = df.groupby(['date_only','Station_No'])['PM2.5'].mean().reset_index()
daily['date_only'] = pd.to_datetime(daily['date_only'])

fig, ax = plt.subplots(figsize=(15,6))
station_colors = {1:'tab:blue', 2:'tab:orange', 3:'tab:red',
                  4:'tab:green', 5:'tab:purple', 6:'tab:brown'}
station_names = {1:'Trạm 1 (ĐHQG)', 2:'Trạm 2 (Giao thông)',
                 3:'Trạm 3 (KCN Tân Bình)', 4:'Trạm 4 (Bình Thạnh)',
                 5:'Trạm 5 (Giao thông)', 6:'Trạm 6 (Giao thông + DC)'}
for s in sorted(daily['Station_No'].unique()):
    sub = daily[daily['Station_No']==s].sort_values('date_only')
    ax.plot(sub['date_only'], sub['PM2.5'], lw=0.8, alpha=0.8,
            color=station_colors[s], label=station_names[s])
ax.axhline(25, color='black', linestyle='--', alpha=0.5, label='Ngưỡng WHO 25')
ax.set_xlabel('Thời gian'); ax.set_ylabel('PM2.5 trung bình ngày (µg/m³)')
ax.set_title('Biến động PM2.5 trung bình ngày tại các trạm')
ax.legend(loc='upper right', ncol=2, fontsize=9)
plt.tight_layout(); plt.show()

Trạm 3 (KCN Tân Bình) có nồng độ cao vượt trội so với các trạm khác
trong gần như toàn bộ thời kỳ. Tất cả các trạm đều có pattern mùa rõ
rệt: PM2.5 cao vào mùa khô (11-4), thấp hơn vào mùa mưa (5-10).

### 2.7.2 PM2.5 theo giờ trong ngày

In [ ]:
hourly = df.groupby('hour')['PM2.5'].agg(['mean','median','std']).reset_index()

fig, ax = plt.subplots(figsize=(12,5))
ax.plot(hourly['hour'], hourly['mean'], marker='o', lw=2, label='Trung bình')
ax.plot(hourly['hour'], hourly['median'], marker='s', lw=2, label='Trung vị')
ax.fill_between(hourly['hour'],
                 hourly['mean']-hourly['std'], hourly['mean']+hourly['std'],
                 alpha=0.2, label='±1 SD')
ax.axhline(25, color='red', linestyle='--', alpha=0.5, label='WHO 25')
ax.set_xlabel('Giờ trong ngày (0-23)')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('PM2.5 theo giờ trong ngày')
ax.set_xticks(range(0,24,2))
ax.legend()
plt.tight_layout(); plt.show()

print(df.groupby('hour_group')['PM2.5'].mean().round(2).sort_values(ascending=False))

PM2.5 đạt đỉnh vào ban đêm và sáng sớm (0-7h), thấp nhất vào ban trưa
(12-15h). Ban đêm khí quyển ổn định, ít đối lưu nên bụi tích tụ ở tầng
thấp. Ban trưa nhiệt độ cao thúc đẩy đối lưu, bụi khuếch tán lên cao.

### 2.7.3 PM2.5 theo ngày trong tuần

In [ ]:
day_names = ['T2','T3','T4','T5','T6','T7','CN']
dow_stats = df.groupby('dow')['PM2.5'].agg(['mean','median']).reset_index()

fig, ax = plt.subplots(figsize=(10,5))
x = dow_stats['dow']
ax.bar(x-0.2, dow_stats['mean'], width=0.4, label='Trung bình', alpha=0.8)
ax.bar(x+0.2, dow_stats['median'], width=0.4, label='Trung vị', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(day_names)
ax.set_xlabel('Ngày trong tuần')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_title('PM2.5 theo ngày trong tuần')
ax.legend()
ax.axhline(25, color='red', linestyle='--', alpha=0.5)
plt.tight_layout(); plt.show()

print(f"PM2.5 TB ngày thường (T2-T6): {df[df['dow']<5]['PM2.5'].mean():.2f}")
print(f"PM2.5 TB cuối tuần (T7-CN) : {df[df['dow']>=5]['PM2.5'].mean():.2f}")

Mức PM2.5 khá đồng đều giữa các ngày trong tuần. Nguồn phát thải chủ
yếu là hoạt động cố định (công nghiệp, sinh hoạt) hơn là giao thông
biến đổi theo tuần.

### 2.7.4 PM2.5 theo tháng

In [ ]:
month_stats = df.groupby('month')['PM2.5'].agg(['mean','std','count']).reset_index()
month_names = ['T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12']

fig, ax = plt.subplots(figsize=(12,5))
yerr = (month_stats['std'] / np.sqrt(month_stats['count'])).values
ax.bar(month_stats['month'], month_stats['mean'].values,
       yerr=yerr, capsize=4, alpha=0.8, color='steelblue', edgecolor='black')
ax.axhline(25, color='red', linestyle='--', alpha=0.5, label='WHO 25')
ax.axvspan(10.5, 12.5, alpha=0.15, color='orange', label='Mùa khô')
ax.axvspan(0.5, 4.5, alpha=0.15, color='orange')
ax.set_xticks(range(1,13)); ax.set_xticklabels(month_names)
ax.set_xlabel('Tháng')
ax.set_ylabel('PM2.5 trung bình (µg/m³)')
ax.set_title('PM2.5 trung bình theo tháng')
ax.legend()
plt.tight_layout(); plt.show()

print(f"PM2.5 TB mùa khô (T11-T4): {df[df['is_dry_season']==1]['PM2.5'].mean():.2f}")
print(f"PM2.5 TB mùa mưa (T5-T10): {df[df['is_dry_season']==0]['PM2.5'].mean():.2f}")

Mùa khô (T11-T4) có PM2.5 cao hơn mùa mưa khoảng 6 µg/m³. Mưa rửa bụi
khỏi không khí; độ ẩm cao mùa mưa làm bụi ngưng tụ và lắng xuống.

# Chương 3. Hồi quy tuyến tính

Chương này trả lời câu hỏi yếu tố nào ảnh hưởng đến PM2.5. Quy trình:

1. Mã hoá biến chỉ số (Station, hour_group)
2. Temporal split theo từng trạm
3. Mô hình đầy đủ + kiểm định
4. So sánh M1/M2/M3/M4 bằng AIC/BIC/Cp/PRESS
5. Stepwise selection
6. Chẩn đoán phần dư
7. Biến đổi Box-Cox
8. So sánh có vs không TSP
9. Phân tích yếu tố ảnh hưởng (standardized coefficients)

## 3.1 Mã hoá biến chỉ số

In [ ]:
df_model = pd.get_dummies(df, columns=['Station_No','hour_group'],
                          prefix=['S','H'], drop_first=True, dtype=int)
station_dummies = [c for c in df_model.columns if c.startswith('S_')]
hour_dummies = [c for c in df_model.columns if c.startswith('H_')]

print('Biến chỉ số trạm:', station_dummies)
print('Biến chỉ số giờ:', hour_dummies)

## 3.2 Temporal split

Dữ liệu là chuỗi thời gian. Random split sẽ trộn quan sát train và test
xen kẽ theo thời gian, gây leakage. Ta sắp xếp theo thời gian rồi lấy
80% đầu làm train, 20% cuối làm test, thực hiện trong từng trạm để cân
bằng giữa các trạm.

In [ ]:
df_sorted = df.sort_values(['Station_No','date']).reset_index()
train_indices, test_indices = [], []
for station in sorted(df_sorted['Station_No'].unique()):
    mask = df_sorted['Station_No'] == station
    n = mask.sum()
    n_train = int(0.8 * n)
    idx_sorted = df_sorted.index[mask].tolist()
    train_indices.extend(df_sorted.loc[idx_sorted[:n_train], 'index'].tolist())
    test_indices.extend(df_sorted.loc[idx_sorted[n_train:], 'index'].tolist())

train_df = df_model.loc[train_indices].reset_index(drop=True)
test_df = df_model.loc[test_indices].reset_index(drop=True)

print(f'Train: {len(train_df):,} quan sát')
print(f'Test : {len(test_df):,} quan sát')
print(f'Khoảng thời gian train: {df.loc[train_indices, "date"].min()} → {df.loc[train_indices, "date"].max()}')
print(f'Khoảng thời gian test : {df.loc[test_indices, "date"].min()} → {df.loc[test_indices, "date"].max()}')

## 3.3 Mô hình đầy đủ

In [ ]:
quant = ['TSP','O3','CO','NO2','SO2','Temperature','Humidity']
time_feat = ['is_weekend','is_dry_season']
all_features = quant + station_dummies + hour_dummies + time_feat

X_train = train_df[all_features]
y_train = train_df['PM2.5']
X_test = test_df[all_features]
y_test = test_df['PM2.5']

X_train_sm = sm.add_constant(X_train).astype(float)
ols_full = sm.OLS(y_train, X_train_sm).fit()
print(ols_full.summary())

## 3.4 Kiểm định đa cộng tuyến (VIF)

In [ ]:
vif_df = pd.DataFrame({
    'Feature': X_train_sm.columns,
    'VIF': [variance_inflation_factor(X_train_sm.values, i) for i in range(X_train_sm.shape[1])]
})
print(vif_df.round(3))
print('\nNgưỡng: VIF > 5 đáng lưu ý; > 10 nghiêm trọng.')

## 3.5 So sánh các mô hình ứng cử

Bốn mô hình có độ phức tạp tăng dần:
- M1: chỉ biến hoá học (TSP, CO, NO2, SO2, O3)
- M2: + khí tượng (Temperature, Humidity)
- M3: + vị trí (5 biến chỉ số trạm)
- M4: + thời gian (4 biến chỉ số giờ, weekend, dry_season)

In [ ]:
def fit_and_compute(X, y, name):
    Xs = sm.add_constant(X).astype(float)
    m = sm.OLS(y, Xs).fit()
    Q, _ = np.linalg.qr(Xs.values)
    h = (Q**2).sum(axis=1)
    e = m.resid.values
    press = float(np.sum((e / (1 - h))**2))
    return {
        'Model': name,
        'p': int(len(m.params)),
        'R²': m.rsquared,
        'R²_Adj': m.rsquared_adj,
        'MSRes': m.mse_resid,
        'AIC': m.aic,
        'BIC': m.bic,
        'PRESS': press,
        'SSRes': float(np.sum(m.resid**2))
    }

results = []
results.append(fit_and_compute(X_train[['TSP','O3','CO','NO2','SO2']], y_train, 'M1: Hoá học'))
results.append(fit_and_compute(X_train[quant], y_train, 'M2: + Khí tượng'))
results.append(fit_and_compute(X_train[quant + station_dummies], y_train, 'M3: + Vị trí'))
results.append(fit_and_compute(X_train, y_train, 'M4: + Thời gian (full)'))

sigma2_full = results[-1]['MSRes']
n_train = len(X_train)
for r in results:
    r['Cp'] = r['SSRes']/sigma2_full - n_train + 2*r['p']

comp = pd.DataFrame(results)[['Model','p','R²','R²_Adj','MSRes','Cp','AIC','BIC','PRESS']]
print(comp.to_string(index=False))

Thêm vị trí (M2 → M3) cải thiện R² đáng kể nhất. M4 là tốt nhất ở mọi
tiêu chí và được chọn làm cơ sở cho các phân tích tiếp theo.

## 3.6 Stepwise selection

Tài liệu (Mục 3.4.2) trình bày stepwise như phương pháp chính thống.
Harrell (2001) và Steyerberg (2009) đã phê bình stepwise có thể gây
p-value inflation và overfitting nên ta kết hợp với AIC/BIC và test set.

In [ ]:
def stepwise_selection(X, y, threshold_in=0.05, threshold_out=0.10, verbose=True):
    included = []
    while True:
        changed = False
        excluded = list(set(X.columns) - set(included))
        if excluded:
            new_pval = pd.Series(index=excluded, dtype=float)
            for col in excluded:
                m = sm.OLS(y, sm.add_constant(X[included+[col]]).astype(float)).fit()
                new_pval[col] = m.pvalues[col]
            if new_pval.min() < threshold_in:
                best = new_pval.idxmin()
                included.append(best); changed = True
                if verbose: print(f'  + {best} (p={new_pval.min():.4g})')
        if included:
            m = sm.OLS(y, sm.add_constant(X[included]).astype(float)).fit()
            pvals = m.pvalues.iloc[1:]
            if pvals.max() > threshold_out:
                worst = pvals.idxmax(); included.remove(worst); changed = True
                if verbose: print(f'  - {worst} (p={pvals.max():.4g})')
        if not changed: break
    return included

selected = stepwise_selection(X_train, y_train)
print(f'\nBiến được chọn ({len(selected)}/{len(X_train.columns)}):')
print(selected)

## 3.7 Mô hình cuối

In [ ]:
X_train_final = sm.add_constant(X_train[selected]).astype(float)
ols_final = sm.OLS(y_train, X_train_final).fit()
print(ols_final.summary())

## 3.8 Chẩn đoán phần dư

In [ ]:
X_test_final = sm.add_constant(X_test[selected]).astype(float)
y_pred = ols_final.predict(X_test_final)
resid = y_test - y_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].scatter(y_pred, resid, s=3, alpha=0.3)
axes[0,0].axhline(0, color='red', linestyle='--')
axes[0,0].set_xlabel('Giá trị dự đoán'); axes[0,0].set_ylabel('Phần dư')
axes[0,0].set_title('(a) Phần dư vs dự đoán')

sm.qqplot(resid, line='45', fit=True, ax=axes[0,1])
axes[0,1].set_title('(b) Q-Q plot phần dư')

axes[1,0].hist(resid, bins=60, edgecolor='black', alpha=0.7)
axes[1,0].set_xlabel('Phần dư'); axes[1,0].set_ylabel('Tần suất')
axes[1,0].set_title('(c) Histogram phần dư')

axes[1,1].scatter(range(len(resid)), resid, s=2, alpha=0.3)
axes[1,1].axhline(0, color='red', linestyle='--')
axes[1,1].set_xlabel('Quan sát test (theo thứ tự thời gian)')
axes[1,1].set_ylabel('Phần dư')
axes[1,1].set_title('(d) Phần dư theo thời gian')

plt.tight_layout(); plt.show()

bp = het_breuschpagan(ols_final.resid, X_train_final.values)
jb = jarque_bera(ols_final.resid)
print(f'Breusch-Pagan: LM = {bp[0]:.2f}, p = {bp[1]:.4g}')
print(f'Jarque-Bera : JB = {jb[0]:.2f}, p = {jb[1]:.4g}')
print(f'  Skewness = {jb[2]:.3f}, Kurtosis = {jb[3]:.3f}')

Phần dư có heteroscedasticity (BP bác bỏ H0) và không chuẩn (skew và
kurt cách xa 0, 3). Đây là vi phạm thường gặp ở dữ liệu môi trường. Với
cỡ mẫu rất lớn (n > 42.000), các ước lượng vẫn nhất quán theo định lý
giới hạn trung tâm.

## 3.9 Biến đổi Box-Cox

Khi phần dư vi phạm giả định chuẩn, có thể thử biến đổi biến phản hồi.
Phương pháp Box-Cox tự động tìm λ tối ưu.

In [ ]:
y_train_pos = y_train + 1
lambda_opt = boxcox_normmax(y_train_pos, method='mle')
print(f'λ tối ưu (MLE): {lambda_opt:.4f}')

if abs(lambda_opt) < 0.1:
    print('λ gần 0: gợi ý biến đổi log')
elif abs(lambda_opt - 0.5) < 0.1:
    print('λ gần 0.5: gợi ý biến đổi sqrt')
else:
    print(f'Box-Cox với λ = {lambda_opt:.3f}')

lambdas = np.linspace(-1, 1.5, 26)
ssres = []
for lam in lambdas:
    if abs(lam) < 1e-6:
        y_trans = np.log(y_train_pos)
    else:
        y_trans = (y_train_pos**lam - 1) / lam
    m_tmp = sm.OLS(y_trans, X_train_final).fit()
    ssres.append(np.sum(m_tmp.resid**2))

plt.figure(figsize=(10,5))
plt.plot(lambdas, ssres, marker='o', lw=2)
plt.axvline(lambda_opt, color='red', linestyle='--', label=f'λ* = {lambda_opt:.3f}')
plt.axvline(0, color='green', linestyle=':', alpha=0.5, label='λ = 0 (log)')
plt.axvline(1, color='gray', linestyle=':', alpha=0.5, label='λ = 1 (no transform)')
plt.xlabel('λ'); plt.ylabel('SSRes(λ)')
plt.title('Box-Cox: SSRes theo λ')
plt.legend(); plt.tight_layout(); plt.show()

## 3.10 So sánh có TSP vs không TSP

TSP là tổng các hạt bụi lơ lửng; về vật lý, TSP bao gồm PM2.5. Đưa TSP
vào mô hình giải thích PM2.5 sẽ cho R² cao nhưng phần nào là "tự giải
thích". Ta xây 2 mô hình:
- Mô hình A (có TSP): mối quan hệ thống kê đầy đủ
- Mô hình B (không TSP): chỉ dùng yếu tố ngoại sinh (khí tượng, vị trí,
  thời gian, các chất ô nhiễm khác)

In [ ]:
results_tsp = []
for name, feats in [
    ('CÓ TSP', selected),
    ('KHÔNG TSP', [f for f in selected if f != 'TSP'])
]:
    X_tr = sm.add_constant(X_train[feats]).astype(float)
    X_te = sm.add_constant(X_test[feats]).astype(float)
    m = sm.OLS(y_train, X_tr).fit()
    yp = m.predict(X_te)
    ss_res = np.sum((y_test - yp)**2)
    ss_tot = np.sum((y_test - y_test.mean())**2)
    r2_test = 1 - ss_res/ss_tot
    rmse = np.sqrt(np.mean((y_test - yp)**2))
    mae = np.mean(np.abs(y_test - yp))
    results_tsp.append({
        'Mô hình': name,
        'R²_train': m.rsquared,
        'R²_Adj': m.rsquared_adj,
        'AIC': m.aic,
        'R²_test': r2_test,
        'RMSE_test': rmse,
        'MAE_test': mae
    })

df_tsp = pd.DataFrame(results_tsp)
print(df_tsp.round(4).to_string(index=False))

Bỏ TSP làm R² test rớt từ 0.73 xuống 0.13, RMSE tăng gần gấp đôi. Như
vậy TSP chứa thông tin chủ đạo về PM2.5 do quan hệ vật lý "bao gồm".
Khi chỉ dùng yếu tố ngoại sinh (loại trừ TSP), mô hình giải thích được
khoảng 25% biến thiên — phản ánh khả năng dự báo trong tình huống
không có thông tin về bụi tổng.

## 3.11 Phân tích yếu tố ảnh hưởng đến PM2.5

Phần này trả lời câu hỏi nghiên cứu chính. Quy trình:
1. Tính standardized coefficients
2. Xếp hạng các yếu tố
3. Phân tích đóng góp theo nhóm

### 3.11.1 Standardized coefficients

Hệ số gốc β có đơn vị khác nhau giữa các biến nên không so sánh trực
tiếp được. Standardized coefficient β_std = β × σ_x / σ_y cho biết số
độ lệch chuẩn y thay đổi khi x tăng 1 độ lệch chuẩn.

In [ ]:
sd_y = y_train.std()
rows = []
for col in ols_final.params.index:
    if col == 'const': continue
    if col in X_train.columns:
        sd_x = X_train[col].std()
        beta_std = ols_final.params[col] * sd_x / sd_y
        rows.append({
            'Biến': col,
            'β_raw': ols_final.params[col],
            'β_std': beta_std,
            '|β_std|': abs(beta_std),
            't-stat': ols_final.tvalues[col],
            'p-value': ols_final.pvalues[col]
        })

df_imp = pd.DataFrame(rows).sort_values('|β_std|', ascending=False).reset_index(drop=True)
df_imp.insert(0, 'Hạng', range(1, len(df_imp)+1))
print(df_imp.round(4).to_string(index=False))

### 3.11.2 Biểu đồ xếp hạng

In [ ]:
fig, ax = plt.subplots(figsize=(11, max(6, len(df_imp)*0.4)))
colors = ['tab:red' if x < 0 else 'tab:green' for x in df_imp['β_std']]
ax.barh(df_imp['Biến'][::-1], df_imp['β_std'][::-1], color=colors[::-1],
        edgecolor='black', alpha=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardized coefficient β_std')
ax.set_title('Mức độ ảnh hưởng đến PM2.5')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

### 3.11.3 Phân tích theo nhóm

In [ ]:
groups = {
    'Chất ô nhiễm khác (TSP, CO, NO2, SO2, O3)': ['TSP','O3','CO','NO2','SO2'],
    'Khí tượng (Nhiệt độ, Độ ẩm)': ['Temperature','Humidity'],
    'Vị trí địa lý (5 trạm)': station_dummies,
    'Giờ trong ngày': hour_dummies,
    'Mùa và tuần': time_feat
}

group_imp = []
for name, cols in groups.items():
    relevant = df_imp[df_imp['Biến'].isin(cols)]
    if len(relevant) > 0:
        sum_abs = relevant['|β_std|'].sum()
        n_sig = (relevant['p-value'] < 0.05).sum()
        group_imp.append({
            'Nhóm': name,
            'Số biến': len(cols),
            'Số biến có ý nghĩa': n_sig,
            'Σ|β_std|': sum_abs,
        })

df_grp = pd.DataFrame(group_imp)
total = df_grp['Σ|β_std|'].sum()
df_grp['% tổng'] = (df_grp['Σ|β_std|'] / total * 100)
df_grp = df_grp.sort_values('Σ|β_std|', ascending=False).reset_index(drop=True)
print(df_grp.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(10,5))
ax.barh(df_grp['Nhóm'][::-1], df_grp['% tổng'][::-1],
        color='steelblue', edgecolor='black', alpha=0.8)
ax.set_xlabel('% tổng |β_std|')
ax.set_title('Đóng góp giải thích PM2.5 theo nhóm yếu tố')
plt.tight_layout(); plt.show()

### 3.11.4 Diễn giải

Thứ tự tác động đến PM2.5:

1. TSP là yếu tố ảnh hưởng mạnh nhất (β_std ≈ 0.89). Phần lớn là tương
   quan vật lý vì PM2.5 là tập con của TSP.
2. Khu công nghiệp Tân Bình (S_3) là yếu tố không gian quan trọng nhất
   (β_std ≈ 0.57). Vùng KCN cao hơn các vùng khác 22-23 µg/m³ ở cùng
   điều kiện khí tượng.
3. Các chất ô nhiễm CO và NO2 có tác động vừa phải. NO2 có hệ số âm
   sau khi kiểm soát các biến khác — hiệu ứng đối thủ giữa các nguồn
   phát thải.
4. Mùa khô có ảnh hưởng dương đáng kể (β_std ≈ 0.07): cùng điều kiện
   khác, PM2.5 mùa khô cao hơn mùa mưa khoảng 2 µg/m³.
5. Khí tượng và giờ trong ngày có tác động nhỏ hơn dự đoán ban đầu khi
   kiểm soát các biến khác.

Câu trả lời cho RQ: yếu tố ảnh hưởng đến PM2.5 chủ yếu là (i) các chất
ô nhiễm cùng nguồn phát thải, (ii) vị trí địa lý (đặc biệt KCN), và
(iii) yếu tố mùa. Khí tượng và giờ trong ngày có ảnh hưởng nhưng nhỏ
hơn.

## 3.12 Đánh giá khả năng dự báo trên test

In [ ]:
y_pred_final = ols_final.predict(X_test_final)
ss_res = np.sum((y_test - y_pred_final)**2)
ss_tot = np.sum((y_test - y_test.mean())**2)
r2_pred = 1 - ss_res/ss_tot
rmse = np.sqrt(np.mean((y_test - y_pred_final)**2))
mae = np.mean(np.abs(y_test - y_pred_final))

print(f'R² train     : {ols_final.rsquared:.4f}')
print(f'R²_Adj train : {ols_final.rsquared_adj:.4f}')
print(f'R² test      : {r2_pred:.4f}')
print(f'Chênh lệch   : {ols_final.rsquared - r2_pred:+.4f}')
print(f'RMSE test    : {rmse:.4f} µg/m³')
print(f'MAE test     : {mae:.4f} µg/m³')

fig, ax = plt.subplots(figsize=(8,8))
ax.scatter(y_test, y_pred_final, s=3, alpha=0.3)
lo, hi = min(y_test.min(), y_pred_final.min()), max(y_test.max(), y_pred_final.max())
ax.plot([lo,hi],[lo,hi], 'r--', label='y = x')
ax.set_xlabel('PM2.5 thực tế'); ax.set_ylabel('PM2.5 dự đoán')
ax.set_title(f'Dự đoán vs thực tế trên test (R² = {r2_pred:.3f})')
ax.legend()
plt.tight_layout(); plt.show()

## 3.13 Tổng kết chương 3

Mô hình OLS với 16 biến giải thích đạt R² test = 0.73 và RMSE = 6.28
µg/m³, vượt KPI đã đặt ra. Câu hỏi nghiên cứu được trả lời: TSP và vị
trí KCN là hai yếu tố ảnh hưởng mạnh nhất, tiếp đến là các chất ô
nhiễm CO, NO2, yếu tố mùa khô, các trạm giao thông, cuối cùng là khí
tượng và giờ trong ngày.

Khuyến nghị: ưu tiên giám sát phát thải tại KCN Tân Bình, giảm bụi
tổng vào mùa khô, kết hợp đo TSP với PM2.5 cho cảnh báo sớm.

# Chương 4. Phân tích TSP để hiểu quan hệ TSP - PM2.5

Chương 3 cho thấy TSP là biến giải thích quan trọng nhất cho PM2.5 với
β_std ≈ 0.89. Tuy nhiên TSP về vật lý bao gồm PM2.5 (TSP ⊇ PM2.5) nên
quan hệ này phần nào là tương quan tự nhiên giữa hai biến cùng đo
khối lượng bụi. Chương này xây mô hình hồi quy với Y = TSP để xem các
yếu tố ngoại sinh ảnh hưởng đến TSP có giống ảnh hưởng đến PM2.5 không.

## 4.1 Đặt vấn đề

TSP và PM2.5 đều đo khối lượng bụi nhưng khác về thành phần:
- PM2.5: hạt có đường kính khí động học < 2.5 µm (hạt mịn, thường từ
  phản ứng hoá học và đốt cháy)
- TSP: tổng các hạt lơ lửng < ~100 µm (bao gồm cả bụi thô từ đường, đất,
  công trình xây dựng)

Vì PM2.5 ⊂ TSP, các yếu tố khí tượng, vị trí, thời gian có thể ảnh
hưởng đến 2 loại bụi theo cách khác nhau. So sánh pattern hệ số giữa
2 mô hình giúp hiểu rõ hơn về nguồn phát thải tại các vị trí khác nhau.

## 4.2 Mô hình Y = TSP

Để tránh leakage, bỏ PM2.5 khỏi danh sách biến giải thích (tương tự
cách bỏ TSP khỏi mô hình PM2.5 ở mục 3.10). Tập biến giải thích: O3,
CO, NO2, SO2, Temperature, Humidity, 5 dummies trạm, 4 dummies giờ,
is_weekend, is_dry_season.

In [ ]:
# Tập biến giải thích cho TSP (loại PM2.5 — vốn không có sẵn trong all_features)
features_tsp = ['O3','CO','NO2','SO2','Temperature','Humidity'] + station_dummies + hour_dummies + time_feat

X_tr_tsp = sm.add_constant(train_df[features_tsp]).astype(float)
X_te_tsp = sm.add_constant(test_df[features_tsp]).astype(float)
y_tr_tsp = train_df['TSP']
y_te_tsp = test_df['TSP']

ols_tsp = sm.OLS(y_tr_tsp, X_tr_tsp).fit()
print(ols_tsp.summary())

## 4.3 Đánh giá khả năng dự báo TSP

In [ ]:
yp_tsp = ols_tsp.predict(X_te_tsp)
r2_tsp = 1 - np.sum((y_te_tsp - yp_tsp)**2) / np.sum((y_te_tsp - y_te_tsp.mean())**2)
rmse_tsp = np.sqrt(np.mean((y_te_tsp - yp_tsp)**2))
mae_tsp = np.mean(np.abs(y_te_tsp - yp_tsp))

print(f'R² train     : {ols_tsp.rsquared:.4f}')
print(f'R²_Adj train : {ols_tsp.rsquared_adj:.4f}')
print(f'R² test      : {r2_tsp:.4f}')
print(f'RMSE test    : {rmse_tsp:.2f} µg/m³')
print(f'MAE test     : {mae_tsp:.2f} µg/m³')
print(f'\nMean TSP: {df["TSP"].mean():.2f}, SD: {df["TSP"].std():.2f}')

So sánh khả năng dự báo của 2 mô hình (đều bỏ biến cùng loại bụi để
tránh leakage):
- Y = PM2.5 (bỏ TSP): R² test ≈ 0.12, RMSE ≈ 11.4 µg/m³
- Y = TSP (bỏ PM2.5): R² test ≈ 0.38, RMSE ≈ 25 µg/m³

TSP dự báo được tốt hơn từ các yếu tố ngoại sinh. Lý do: TSP nắm bắt
được phát thải vật chất tổng quát, trong khi PM2.5 phụ thuộc nhiều
vào quá trình hình thành hạt mịn thứ cấp (phản ứng hoá học giữa các
khí trong khí quyển) — quá trình này không thể nắm bắt bằng các biến
hiện có.

## 4.4 So sánh standardized coefficients giữa 2 mô hình

In [ ]:
def get_beta_std(model, X_train_df, y_train_series):
    sd_y = y_train_series.std()
    rows = []
    for col in model.params.index:
        if col == 'const': continue
        if col in X_train_df.columns:
            sd_x = X_train_df[col].std()
            rows.append({
                'Biến': col,
                'β_std': model.params[col] * sd_x / sd_y,
                'p_value': model.pvalues[col]
            })
    return pd.DataFrame(rows)

beta_pm = get_beta_std(ols_final, X_train, y_train)
beta_pm = beta_pm.rename(columns={'β_std': 'β_std_PM25', 'p_value': 'p_PM25'})

beta_tsp = get_beta_std(ols_tsp, train_df[features_tsp], y_tr_tsp)
beta_tsp = beta_tsp.rename(columns={'β_std': 'β_std_TSP', 'p_value': 'p_TSP'})

compare = beta_pm.merge(beta_tsp, on='Biến', how='outer')
compare['Đảo dấu'] = (compare['β_std_PM25'] * compare['β_std_TSP'] < 0).map({True: 'Có', False: ''})
compare = compare.sort_values('β_std_PM25', key=lambda x: x.abs(), ascending=False)
print(compare.round(4).to_string(index=False))

## 4.5 Biểu đồ so sánh

In [ ]:
compare_plot = compare.dropna(subset=['β_std_PM25','β_std_TSP'])
# Sắp xếp theo |β_std_PM25|
compare_plot = compare_plot.sort_values('β_std_PM25', key=lambda x: x.abs(), ascending=True)

fig, ax = plt.subplots(figsize=(11, max(6, len(compare_plot)*0.4)))
x = np.arange(len(compare_plot))
w = 0.4
ax.barh(x - w/2, compare_plot['β_std_PM25'], w, label='Mô hình PM2.5', color='tab:blue', alpha=0.8, edgecolor='black')
ax.barh(x + w/2, compare_plot['β_std_TSP'], w, label='Mô hình TSP', color='tab:orange', alpha=0.8, edgecolor='black')
ax.set_yticks(x)
ax.set_yticklabels(compare_plot['Biến'])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardized coefficient β_std')
ax.set_title('So sánh tác động của các yếu tố lên PM2.5 và TSP')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

## 4.6 Diễn giải các khác biệt

Phát hiện chính:

1. KCN Tân Bình (S_3): β_std = +0.57 với PM2.5, nhưng β_std ≈ -0.48 với
   TSP. KCN phát thải nhiều hạt mịn (PM2.5) hơn bụi thô. Có thể do các
   hoạt động đốt nhiên liệu công nghiệp, hàn cắt kim loại, sản xuất hoá
   chất - các nguồn tạo nhiều hạt mịn.

2. Trạm ĐHQG (Thủ Đức, mức tham chiếu): PM2.5 thấp nhưng TSP cao. Khu
   ngoại ô có đường đất, công trình xây dựng, đất trống → bụi thô (đường
   kính lớn) chiếm đa số.

3. O3 và SO2: tác động dương mạnh hơn với TSP so với PM2.5. Có thể liên
   quan đến các nguồn đốt nhiên liệu mạnh phát ra cả ozon thứ cấp và
   bụi thô.

4. Mùa khô: dương cho cả 2 nhưng mạnh hơn với TSP. Mùa mưa rửa được bụi
   thô hiệu quả hơn bụi mịn (vì hạt thô dễ bị nước mưa cuốn xuống).

5. Trạm Quận 3 (S_5): pattern khác biệt giữa 2 mô hình → cấu trúc nguồn
   phát thải tại đây không đồng nhất với các trạm giao thông khác.

## 4.7 Kết luận chương 4

Phân tích cho thấy PM2.5 và TSP không đồng nhất về nguồn phát thải,
đặc biệt tại các vị trí khác nhau. KCN Tân Bình là điểm nóng PM2.5
nhưng không phải điểm nóng TSP — phản ánh đặc trưng phát thải hạt mịn
của hoạt động công nghiệp.

Ý nghĩa thực tiễn:

- Giám sát môi trường cần đo cả 2 chỉ số. Chỉ đo TSP có thể bỏ sót ô
  nhiễm PM2.5 nghiêm trọng tại các vùng công nghiệp.
- Khả năng dự báo PM2.5 từ yếu tố ngoại sinh hạn chế (R² ≈ 0.12) trong
  khi TSP dự báo khá hơn (R² ≈ 0.38) — phản ánh bản chất PM2.5 phụ
  thuộc nhiều vào phản ứng hoá học khí quyển phức tạp.
- TSP không thể thay thế PM2.5 như một proxy đo nhanh vì pattern phát
  thải khác nhau, đặc biệt tại các nguồn công nghiệp.

# Tổng kết

## Trả lời câu hỏi nghiên cứu

Câu hỏi: yếu tố nào ảnh hưởng đến PM2.5 tại TP.HCM?

Trả lời, xếp theo mức độ ảnh hưởng (β_std):

- Hạng 1: TSP (≈0.89) — bụi tổng, có quan hệ vật lý với PM2.5
- Hạng 2: KCN Tân Bình (≈0.57) — vị trí có PM2.5 cao vượt trội
- Hạng 3: NO2 (≈−0.11) — chất ô nhiễm cùng nguồn, hệ số âm sau khi kiểm soát biến khác
- Hạng 4: CO (≈0.08) — chất ô nhiễm cùng nguồn
- Hạng 5: Mùa khô (≈0.07) — yếu tố thời gian mạnh nhất
- Hạng 6: Các trạm giao thông (≈0.05–0.06)
- Hạng 7: Nhiệt độ, độ ẩm (≈0.04)
- Hạng 8: Giờ trong ngày (≈0.02–0.03)

## Kết quả mô hình

Mô hình cuối Y=PM2.5 (có TSP) đạt R²_Adj = 0.68, R² test = 0.73, RMSE
= 6.28 µg/m³ — đạt mọi KPI. Mô hình không TSP có R² test = 0.13.

Chương 4 mở rộng phân tích Y=TSP (bỏ PM2.5) cho thấy TSP và PM2.5 có
pattern nguồn phát thải khác nhau, đặc biệt KCN Tân Bình là điểm nóng
PM2.5 nhưng không phải điểm nóng TSP. Điều này phản ánh đặc trưng phát
thải hạt mịn của hoạt động công nghiệp.

## Đóng góp về phương pháp

So với cách làm thông thường, bài có một số điểm cải tiến: temporal
split thay vì random split để tránh leakage chuỗi thời gian; imputation
theo từng trạm thay vì global median; feature engineering thời gian
(hour_group, is_weekend, is_dry_season); so sánh có/không TSP để tách
bạch tương quan vật lý và yếu tố ngoại sinh; standardized coefficients
để xếp hạng các biến có đơn vị khác nhau.

## Hạn chế

Mô hình có TSP cho R² cao nhưng không phải là mô hình nhân quả thuần
do TSP và PM2.5 có quan hệ vật lý bao trùm. Mô hình không TSP có hiệu
năng thấp, gợi ý cần thêm các biến giải thích như gió, mây, hoạt động
công nghiệp cụ thể. Phần dư có dấu hiệu heteroscedasticity và tự tương
quan thời gian — cần ARIMA hoặc mô hình không gian-thời gian để xử lý
triệt để.

## Khuyến nghị

Ưu tiên giảm phát thải tại KCN Tân Bình; tăng cường biện pháp giảm
bụi tổng (vì TSP có quan hệ trực tiếp với PM2.5); chủ động phòng chống
ô nhiễm vào mùa khô (T11-T4); đo song song TSP với PM2.5 để có cảnh
báo sớm.